# Alt 2267 — Parallell CPU-session Block 1/4 (Google Colab High-RAM)

1. Välj **CPU + Mycket RAM-minne (High-RAM)** (inte GPU).
2. Kör Cell 1, Cell 2 och Cell 3. Plan och fröintervall läses automatiskt ur `notebooks/alt2267_plan.json`.

In [ ]:
# Cell 1: Kontrollera CPU-kärnor och minne
!nproc && free -h

In [ ]:
# Cell 2: Hämta senaste koden från GitHub och bygg prov2008 i release-läge
!rm -rf /content/hcsim-colab && git clone --depth 1 https://github.com/Smurfen1/hcsim-colab.git /content/hcsim-colab
!if ! command -v cargo >/dev/null 2>&1; then curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal; fi
!export PATH="$HOME/.cargo/bin:$PATH" && cd /content/hcsim-colab/n10/hcsim && cargo build --release --bin prov2008

In [ ]:
# Cell 3: Kör Alt 2267 Block 1/4 med live-utskrift och automatisk nedladdning
BLOCK_IDX = 0
import json, os, subprocess, sys
plan = json.load(open('/content/hcsim-colab/notebooks/alt2267_plan.json', encoding='utf-8'))
blk = plan['block'][BLOCK_IDX]
cmd_name = plan['kommando']
fron = blk['fron']
inkor = plan['inkor']
ar = plan['ar']
json_ut = '/content/' + blk['json_fil']
md_ut = '/content/' + blk['md_fil']
env = os.environ.copy()
env['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + env.get('PATH', '')
nproc = str(os.cpu_count() or 8)
cmd = ['./target/release/prov2008', cmd_name, '--fron', fron, '--inkor', str(inkor), '--ar', str(ar), '--tradar', nproc, '--ut', md_ut, '--ra-ut', json_ut]
print('Kör block', blk['id'], 'gren=', plan['gren'], 'cmd=', ' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd='/content/hcsim-colab/n10/hcsim', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
rc = p.wait()
assert rc == 0, f'prov2008 avslutade med felkod {rc}'
from google.colab import files
files.download(json_ut)
